# Raytracing ray image

`batcamp` also provides an octree-aware ray tracer. This notebook follows the same pooch-fetched solar-corona dataset used in the larger quick start example, then builds one line-of-sight image: download and cache the file, build the octree objects, define one camera plane of parallel rays, render one image, and then inspect how many cells each ray crossed.


## Load one pooch-cached dataset and build the raytracing objects

This example uses the same solar-corona file as the second part of quick_start.ipynb, fetched from a larger archive with `pooch.retrieve(...)`. The file is downloaded once and then reused from the local pooch cache. Any of the provided files can be used here: `Octree.from_ds(ds)` reconstructs the appropriate tree directly from the dataset.


In [ ]:
from batread import Dataset
import matplotlib.pyplot as plt
import numpy as np
import pooch

from batcamp import Octree, OctreeInterpolator, OctreeRayTracer
from batcamp.constants import XYZ_VARS

[sc_file] = pooch.retrieve(
    url="https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz",
    known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136",
    progressbar=True,
    processor=pooch.Untar(
        members=["run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt"],
    ),
)
print(f"{sc_file=}")

ds = Dataset.from_file(sc_file)
print(ds)

tree = Octree.from_ds(ds)
interp = OctreeInterpolator(tree, ds["Rho [g/cm^3]"])
tracer = OctreeRayTracer(tree)

print(tree)
print(interp)
print(tracer)


## Define an image plane

To make an image, image ray origins and directions must be provided. Here, the rays all start at one upstream $x$ position and travel in the $+x$ direction, so the ray origins form one `(ny, nz, 3)` array while the direction is one shared 3-vector.

In [ ]:

y, z = np.meshgrid(
    np.linspace(-200,200, 511), 
    np.linspace(-200,200, 511), indexing="xy")

x = np.full_like(y, -200)

origins = np.stack((x, y, z), axis=-1)

directions = np.array([1.0, 0.0, 0.0], dtype=float)

print(f"origin grid shape: {origins.shape}")
print(f"shared direction shape: {directions.shape}")

## Compute the synthetic image
This call forms one line-of-sight density image through the solar-corona snapshot.  `trilinear_image` traces the provided rays through the octree, evaluates the trilinearly interpolated density inside the crossed cells, and integrates that density along the path. The returned `rho_los` array has the same `y, z` shape as the input ray grid, so each image pixel corresponds directly to one ray launched from the image plane.

In [ ]:
rho_los, cell_counts = tracer.trilinear_image(interp, origins, directions)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
mesh = ax.pcolormesh(y, z, rho_los, norm="log")
ax.set_xlabel("y")
ax.set_ylabel("z")
ax.set_title("Solar-corona ray-integrated density")
fig.colorbar(mesh, ax=ax, label="line integral of Rho [g/cm^3]")
plt.show()


## Inspect the returned traversal counts

The same `trilinear_image` call also returns one per-ray cell-count image. Plotting those counts can give a low-level view of the octree structure. Of note here is the vertical stripe at $y=0$ this is a consequence of these highly symmetric rays crossing fewer cells in the octree; the stripe is not an artefact.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
mesh = ax.pcolormesh(y, z, cell_counts)
ax.set_xlabel("y")
ax.set_ylabel("z")
ax.set_title("Traversed octree cells per ray")
fig.colorbar(mesh, ax=ax, label="cell count")
plt.show()

## Inspect ray bundle

The rays do not have to be collapsed immediately into an image. `trace(...)` returns one packed `TracedRays` object containing the crossed cell ids and crossing times for each ray in the bundle. That lower-level representation is useful when you want to inspect the traced paths directly or reuse them for other per-segment calculations.

In [ ]:
segments = tracer.trace(origins, directions)
print(segments.shape)


In [ ]:
row_id = origins.shape[0] // 2  # Draw center row of rays where z is zero
row_segments = segments[row_id, ::16]  # Subsample rays for better visibility in the plot

fig, ax = plt.subplots()

# Draw rays from their origin to the right edge of the plot.
for ray_id, origin_xyz in enumerate(row_segments.origins):
    ax.plot(
        [origin_xyz[0], 250],
        [origin_xyz[1], origin_xyz[1]],
        "k->",
        zorder=0,
        label="Rays" if ray_id == 0 else None,
        markevery=2,
    )

# Draw the active ray segments, i.e. the ray segments that intersect the octree cells.
for ray_id in range(row_segments.size):
    ray_xyz = row_segments.xyz(ray_id)
    ax.plot(
        ray_xyz[:, 0],
        ray_xyz[:, 1],
        "|-",
        zorder=2,
        linewidth=2,
        label="Segments" if ray_id == 0 else None,
    )

# Draw central cavity
ax.add_patch(plt.Circle((0.0, 0.0), 1.0, color="darkgrey", label="$R < 1$", zorder=3, alpha=0.5))
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Ray segments in $xy$ plane")
ax.set_aspect("equal")
ax.legend()
plt.show()

In the plot above, the central ray at $y=0$ can be seen to terminate around $x=1$. This is because the rays do not continue across the inner domain boundary at $R=1$. The following example draws the rays which intersect the central cavity and some nearby rays.

In [ ]:
hits_cavity = segments.origins[..., 1] ** 2 + segments.origins[..., 2] ** 2 <= 9.0
in_z0_plane = np.isclose(segments.origins[..., 2], 0.0)
selected_segments = segments[hits_cavity & in_z0_plane]  # use & for both

fig, ax = plt.subplots()

# Draw rays from their origin to the right edge of the plot.
for ray_id, origin_xyz in enumerate(selected_segments.origins):
    ax.plot(
        [origin_xyz[0], 250],
        [origin_xyz[1], origin_xyz[1]],
        "k->",
        zorder=0,
        label="Rays" if ray_id == 0 else None,
        markevery=2,
    )

# Draw the active ray segments, i.e. the ray segments that intersect the octree cells.
for ray_id in range(selected_segments.size):
    ray_xyz = selected_segments.xyz(ray_id)
    ax.plot(
        ray_xyz[:, 0],
        ray_xyz[:, 1],
        "|-",
        zorder=2,
        linewidth=2,
        label="Segments" if ray_id == 0 else None,
    )

# Draw central cavity
ax.add_patch(plt.Circle((0.0, 0.0), 1.0, color="darkgrey", label="$R<1$", zorder=3, alpha=0.5))
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Zoom-in of ray segments in $xy$ plane")
ax.set_xlim(-4,4)
# ax.set_ylim(-4,4)
ax.set_aspect("equal")
ax.legend()
plt.show()